## DGM European Option Pricing
This notebook applies the DGM method to solve the Black-Scholes PDE for a simple European option. 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
from torch.distributions.normal import Normal
import copy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.init() 

### Hyperparameters and Constants

In [ ]:
# Option Parameters
r = 0.05           # risk-free rate
sigma = 0.2        # volatility
K = 100.0          # strike
T = 1.0            # maturity
S_max = 200.0      # maximum price used in domain
S_min = 0.0

In [ ]:
# Neural network training parameters
no_hidden = 4       # number of hidden layers
no_neurons = 64     # number of neurons per hidden layer
lr = 1e-3          # learning rate
no_epochs = 2000   # number of training epochs
no_loops = 10 # number of iterations before resampling
bd_batch_size = 2**8  # batch size for sampling
int_batch_size = 2**10  # batch size for sampling

In [ ]:
# Loss weights
lambda_pde = 1.0
lambda_term = 1.0
lambda_bc = 1.0

### Neural Network Model

In [ ]:
class DGMNet(nn.Module):
    def __init__(self, n_hidden, n_neurons):
        super(DGMNet, self).__init__()
        layers = []
        input_dim = 2  # (t, S) as inputs
        output_dim = 1 # V(t, S) as output
        
        # Input layer
        layers.append(nn.Linear(input_dim, n_neurons))
        layers.append(nn.Tanh())
        
        # Hidden layers
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        
        # Output layer
        layers.append(nn.Linear(n_neurons, output_dim))
        #layers.append(nn.SiLU())
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, t, s):
        inp = torch.cat((t, s), dim=1)
        return self.model(inp)

### Boundary Conditions

In [ ]:
def boundary_upper(t):
    return S_max - K * torch.exp(-r * t)

In [ ]:
def boundary_lower(t):
    return 0.0

In [ ]:
def terminal(S):
    return torch.max(S - K, torch.zeros_like(S))

### Loss Function

In [ ]:
def pde_residual(net, t, s):
    # Enable PyTorch's autograd for second derivative w.r.t. s
    t.requires_grad = True
    s.requires_grad = True
    
    V = net(t, s)
    
    # First derivatives
    dV_dt = torch.autograd.grad(V, t, grad_outputs=torch.ones_like(V), 
                                retain_graph=True, create_graph=True)[0]
    dV_dS = torch.autograd.grad(V, s, grad_outputs=torch.ones_like(V),
                                retain_graph=True, create_graph=True)[0]
    
    # Second derivative
    d2V_dS2 = torch.autograd.grad(dV_dS, s, grad_outputs=torch.ones_like(dV_dS),
                                  retain_graph=True, create_graph=True)[0]
    
    # Black-Scholes PDE
    pde = dV_dt + 0.5 * sigma**2 * s**2 * d2V_dS2 + r * s * dV_dS - r * V
    
    return pde

In [ ]:
def boundary_loss(net, t):
    # S=0 boundary
    s_zeros = torch.zeros_like(t)
    V_0 = net(t, s_zeros)
    bc_dn = boundary_lower(t)
    loss_bc_0 = torch.mean((V_0 - bc_dn)**2)
    
    # S=S_max boundary
    s_upper = torch.ones_like(t) * S_max
    V_up = net(t, s_upper)
    bc_up = boundary_upper(t)
    loss_bc_up = torch.mean((V_up - bc_up)**2)
    
    return loss_bc_0 + loss_bc_up

In [ ]:
def terminal_loss(net, s_terminal):
    tT = torch.ones_like(s_terminal) * T
    V_T = net(tT, s_terminal)
    payoff_T = terminal(s_terminal)
    return torch.mean((V_T - payoff_T)**2)

### Training Loop

In [ ]:
def train_model(model, optimizer, no_epochs, int_batch_size, 
    bd_batch_size, device=device, patience=100, alpha=0.5
):
    # Record of various losses
    losses = {'res': [], 'bc': [], 'term': [], 'total': []}

    # Track the best (lowest) total loss and corresponding model weights
    lowest_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    
    # Track epochs since last improvement
    epochs_since_improvement = 0

    pbar = tqdm(range(no_epochs))
    for epoch in pbar:
        
        
        # Sample interior points for PDE residual
        t_int = torch.rand(int_batch_size, 1, device=device) * T
        s_int = torch.rand(int_batch_size, 1, device=device) * (S_max - S_min) + S_min
        s_terminal = torch.rand(bd_batch_size, 1, device=device) * (S_max - S_min) + S_min

        for i in range(no_loops):
            optimizer.zero_grad()
        
            # PDE residual
            with torch.backends.cudnn.flags(enabled=False):
                pde_val = pde_residual(model, t_int, s_int)
            loss_pde = torch.mean(pde_val**2)
            
            # Boundary points
            t_bd = torch.rand(bd_batch_size, 1, device=device) * T
            loss_bc = boundary_loss(model, t_bd)
            
            # Terminal condition
            
            loss_term = terminal_loss(model, s_terminal)
            
            # Combine losses
            loss = (lambda_pde * loss_pde 
                    + lambda_bc * loss_bc
                    + lambda_term * loss_term)
            
            # Backpropagation
            loss.backward()
            optimizer.step()
        
            # Store losses
            losses['res'].append(loss_pde.item())
            losses['bc'].append(loss_bc.item())
            losses['term'].append(loss_term.item())
            losses['total'].append(loss.item())
        
        # Check if this is the best model so far
        if loss.item() < lowest_loss:
            lowest_loss = loss.item()
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_since_improvement = 0  # reset patience counter
        else:
            epochs_since_improvement += 1
            
            # If we exceeded the patience threshold, reduce the LR
            if epochs_since_improvement >= patience:
                for param_group in optimizer.param_groups:
                    old_lr = param_group['lr']
                    new_lr = old_lr * alpha
                    param_group['lr'] = new_lr
                epochs_since_improvement = 0  # reset counter after LR reduction
                print("New LR: ", new_lr)
        
        # Update progress bar description
        pbar.set_description(
            f"Epoch [{epoch+1}/{no_epochs}] | Loss={loss.item():.6f} "
            f"(PDE={loss_pde.item():.6f}, BC={loss_bc.item():.6f}, Term={loss_term.item():.6f})"
        )
    
    # Load the best model weights
    best_model = copy.deepcopy(model)
    best_model.load_state_dict(best_model_weights)
    
    return best_model, losses

In [ ]:
model = DGMNet(no_hidden, no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
best_model, losses = train_model(model, optimizer, no_epochs, int_batch_size, bd_batch_size)

### Plot Losses

In [ ]:
def plot_losses(losses, filename):
    epochs = np.arange(len(losses['res'])) + 1
    plt.figure(figsize=(15, 10))
    plt.plot(epochs, losses['res'],  color='black', linestyle=':',  label='Residual (PDE)')
    plt.plot(epochs, losses['bc'],   color='black', linestyle='--', label='Boundary')
    plt.plot(epochs, losses['term'], color='black', linestyle='-.', label='Terminal')
    #plt.yscale('log')
    plt.ylim([0, 1000])
    plt.xlabel('Epoch * Loops')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
filename = 'dgm_bs_loss_withr.png'
plot_losses(losses, filename)

### Test Model

In [ ]:
def norm_cdf(x):
    return 0.5 * (1 + torch.erf(x / torch.sqrt(torch.tensor(2.0))))

def black_scholes(S, K, T, r, sigma, option_type='call', device=device):
    S = S.to(device)
    K = K.to(device)
    T = T.to(device)
    r = r.to(device)
    sigma = sigma.to(device)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)

    if option_type == 'call':
        return S * norm_cdf(d1) - K * torch.exp(-r * T) * norm_cdf(d2)
    elif option_type == 'put':
        return K * torch.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
    else:
        raise ValueError("option_type must be either 'call' or 'put'")

In [ ]:
def evaluate_solutions_on_grid(model, t_vals, s_vals, T, K, r, sigma, device=device):
    
    t_grid, s_grid = torch.meshgrid(t_vals, s_vals, indexing='ij')
    t_grid = t_grid.to(device)
    s_grid = s_grid.to(device)
    
    # Evaluate DGM model
    model.eval()
    with torch.no_grad():
        # Flatten for batch evaluation
        t_flat = t_grid.reshape(-1, 1)
        s_flat = s_grid.reshape(-1, 1)
        V_dgm_flat = model(t_flat, s_flat)
        V_dgm = V_dgm_flat.view(t_grid.shape)
    
    
    # Evaluate Black–Scholes
    Kt = torch.full_like(t_flat, K)
    rt = torch.full_like(t_flat, r)
    sigmat = torch.full_like(t_flat, sigma)
    deltat = T - t_flat
    V_bs_flat = black_scholes(s_flat, Kt, deltat, rt, sigmat)
    V_bs = V_bs_flat.view(t_grid.shape)
    
    return t_grid, s_grid, V_dgm, V_bs

In [ ]:
Nt, Ns = 50, 80
t_test = torch.linspace(0.0, T, Nt)
s_test = torch.linspace(0.0, S_max, Ns)
    
t_grid, s_grid, V_dgm, V_bs = evaluate_solutions_on_grid(
    model=best_model,
    t_vals=t_test,
    s_vals=s_test,
    T=T, K=K, r=r, sigma=sigma,
    device=device
)

In [ ]:
intrinsic = terminal(s_grid[0,:])

In [ ]:
def plot_timeslices(s_grid, V_dgm, V_bs, intrinsic, timeslices, filename):
    # Create a 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()  # Flatten so we can iterate easily

    for i, t in enumerate(timeslices):
        ax = axes[i]

        # Plot the intrinsic value if the slice is -1
        if t == 49:
            ax.plot(s_grid[-1, :].cpu(), intrinsic.cpu(), 
                    color='black', linestyle='-.', label='Intrinsic')
            ax.set_title("Intrinsic Value")
        
        # Plot DGM
        ax.plot(
            s_grid[t, :].cpu(),
            V_dgm[t, :].cpu(),
            color='black',
            marker='+',
            linestyle='none',
            label='DGM'
        )
        # Plot BS (markers only, no connecting lines)
        ax.plot(
            s_grid[t, :].cpu(),
            V_bs[t, :].cpu(),
            color='black',
            marker='x',
            linestyle='none',
            label='BS'
        )
        # Label with the time to maturity (t/50)
        ax.set_title(f"Time to maturity = {(50-t-1)/50:.2f} year")

        # Set axis labels
        ax.set_xlabel('Stock Price')
        ax.set_ylabel('Option Price')
        ax.legend()

    # Make layout nice and tight
    plt.tight_layout()

    # Save figure at 300 dpi
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
timeslices = [0, 15, 30, 49]
plot_timeslices(s_grid, V_dgm, V_bs, intrinsic, timeslices, "dgm_option_slices_withr.png")